# Trivialised Diffusion With CSPVNet

This notebook mirrors the style of `continious.ipynb`, but now for the simplified TDM class.

We keep the scope intentionally small:
- use a real MP-20 CSP batch
- sample noisy fractional coordinates `f_t` and noisy velocities `v_t`
- compute the TDM velocity target from the forward process
- train `CSPVNet` only on the `v` head

So this is a clean **TDM + CSPVNet** training loop, not yet the full KLDM paper pipeline.

## Theory

For the simplified trivialised diffusion, we sample:

- velocity:
  \[
  v_t = \alpha_v(t) v_0 + \sigma_v(t) \epsilon_v
  \]
- wrapped displacement:
  \[
  r_t = \mathrm{wrap}(\mu_r(t, v_0, v_t) + \sigma_r(t) \epsilon_r)
  \]
- noisy fractional coordinates:
  \[
  f_t = \mathrm{wrap}(f_0 + r_t)
  \]

The velocity training target is split into two terms, like in KLDM:

1. the Gaussian velocity score
2. the wrapped-normal position contribution

and the final target is their sum.

In this notebook we let `TrivialisedDiffusion.score_target(...)` build that target directly from the sampled forward variables.

In [ ]:
from pathlib import Path

import torch
from torch import nn

from kldm.data import CSPTask, resolve_data_root
from kldm.diffusionModels.trivialized_diffusion import TrivialisedDiffusion
from kldm.scoreNetwork.scoreNetwork import CSPVNet
from kldm.distribution.uniform import sample_uniform

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root = resolve_data_root()

diffusion = TrivialisedDiffusion().to(device)
root, device, diffusion

## 1. Load A Real CSP Batch

In [ ]:
batch = next(
    iter(CSPTask().dataloader(root=root, split='train', batch_size=2, shuffle=False, download=True))
).to(device)

print('num graphs:', batch.num_graphs)
print('pos shape  :', tuple(batch.pos.shape))
print('h shape    :', tuple(batch.h.shape))
print('l shape    :', tuple(batch.l.shape))
print('\nfirst 3 fractional coordinates:')
print(batch.pos[:3])

## 2. Sample The TDM Forward Kernel

In [ ]:
t_graph = sample_uniform(lb=diffusion.eps, size=(batch.num_graphs, 1), device=device)
t_node = t_graph[batch.batch].squeeze(-1)

f_t, v_t, epsilon_v, epsilon_r, r_t = diffusion.forward_sample(t=t_node, f0=batch.pos)
target_v = diffusion.score_target(t=t_node, epsilon_v=epsilon_v, r_t=r_t, v_t=v_t)

print('t_graph:', t_graph)
print('\nOriginal positions (first 3 atoms):')
print(batch.pos[:3])
print('\nNoisy positions f_t (first 3 atoms):')
print(f_t[:3])
print('\nNoisy velocities v_t (first 3 atoms):')
print(v_t[:3])
print('\nVelocity target (first 3 atoms):')
print(target_v[:3])
print('\nWrapped position range:')
print(float(f_t.min()), float(f_t.max()))

## 3. Build CSPVNet

We use the real score network and only keep its `v` head active. The lattice `l` and atom types `h` are passed as conditioning inputs, while the network predicts the TDM velocity target.

In [ ]:
def dsm_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return ((pred - target) ** 2).mean()


model = CSPVNet(
    hidden_dim=128,
    num_layers=4,
    h_dim=118,
    pred_v=True,
    pred_l=False,
    pred_h=False,
    smooth=False,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model

## 4. One CSPVNet Training Step

In [ ]:
batch = next(
    iter(CSPTask().dataloader(root=root, split='train', batch_size=8, shuffle=True, download=True))
).to(device)

t_graph = sample_uniform(
    lb=diffusion.eps,
    size=(batch.num_graphs, 1),
    device=device,
)

t_node = t_graph[batch.batch].squeeze(-1)

f_t, v_t, epsilon_v, epsilon_r, r_t = diffusion.forward_sample(t=t_node, f0=batch.pos)
target_v = diffusion.score_target(t=t_node, epsilon_v=epsilon_v, r_t=r_t, v_t=v_t)

scores = model(
    t=t_graph,
    pos=f_t,
    v=v_t,
    h=batch.h,
    l=batch.l,
    node_index=batch.batch,
    edge_node_index=batch.edge_node_index,
)
pred_v = scores['v']
loss = dsm_loss(pred_v, target_v)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print('pred_v shape :', tuple(pred_v.shape))
print('target shape :', tuple(target_v.shape))
print('loss         :', float(loss))

## 5. Simple Training Loop

This is the same training idea as the continuous notebook: sample a batch, draw random times, sample noisy states from the forward kernel, compute the exact training target, and regress the network onto that target.

In [ ]:
train_loader = CSPTask().dataloader(root=root, split='train', batch_size=16, shuffle=True, download=True)
num_epochs = 5
print_every = 20

for epoch in range(num_epochs):
    running_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = batch.to(device)

        t_graph = sample_uniform(lb=diffusion.eps, size=(batch.num_graphs, 1), device=device)
        t_node = t_graph[batch.batch].squeeze(-1)

        f_t, v_t, epsilon_v, epsilon_r, r_t = diffusion.forward_sample(t=t_node, f0=batch.pos)
        target_v = diffusion.score_target(t=t_node, epsilon_v=epsilon_v, r_t=r_t, v_t=v_t)

        scores = model(
            t=t_graph,
            pos=f_t,
            v=v_t,
            h=batch.h,
            l=batch.l,
            node_index=batch.batch,
            edge_node_index=batch.edge_node_index,
        )
        pred_v = scores['v']
        loss = dsm_loss(pred_v, target_v)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss)

        if step % print_every == 0:
            print(f'epoch={epoch:02d} step={step:03d} loss={float(loss):.6f}')

    mean_loss = running_loss / (step + 1)
    print(f'epoch={epoch:02d} mean_loss={mean_loss:.6f}')